# DeepSeek-OCR test — dict_alif and gojri_adbiyaat

Self-hosted, full prompt control (unlike the community Gradio demos, which only
expose fixed task buttons). Runtime: **T4 GPU** (Runtime > Change runtime type > T4).

**Steps:**
1. Run the install cell (first run only, ~2-3 min).
2. Run the upload cell, and when prompted, upload both
   `dict_alif.png` and `gojri_adbiyaat.png` from your local
   `data/gold/images_cropped/` folder.
3. Run the remaining cells in order.
4. Copy the printed output for each page back to Claude.


In [ ]:
# transformers is pinned, not left latest: DeepSeek-OCR's remote code
# (modeling_deepseekv2.py) imports transformers.models.llama.modeling_llama.
# LlamaFlashAttention2 directly. That class was removed when transformers
# refactored attention internals (a change shipped after DeepSeek-OCR was
# published), so an unpinned install breaks with:
#   ImportError: cannot import name 'LlamaFlashAttention2' ...
# 4.46.3 is the last release before that refactor and is confirmed to work.
!pip install -q transformers==4.46.3 torch accelerate
# DeepSeek-OCR's custom modeling code (modeling_deepseekocr.py, deepencoder.py)
# imports these directly - confirmed by reading the actual source files rather
# than guessing, since there's no requirements.txt in the model repo.
!pip install -q addict easydict einops torchvision
# flash-attn is skipped deliberately, not just optionally: confirmed on a real
# T4 Colab run (2026-08-05) that it isn't preinstalled and reliably fails/takes
# too long to build from source there (no matching prebuilt wheel for T4's CUDA
# toolchain). The model-loading cell below always falls back to eager attention
# in that case anyway, so attempting the build here only wastes install time.


In [ ]:
from google.colab import files
print("Upload dict_alif.png and gojri_adbiyaat.png (from data/gold/images_cropped/ in the repo)")
uploaded = files.upload()
print("Uploaded:", list(uploaded.keys()))


In [ ]:
# Canonical prompt, prompts/ocr_transcription_v1.txt, embedded verbatim so this
# run uses the exact same instructions as every other model already tested.
PROMPT_V1 = "You are transcribing a single page image for a Gojri-language preservation project. Gojri is an Indo-Aryan language spoken by Gujjar communities in Pakistan and India. It is written in the Nastaliq (Perso-Arabic) script and shares that script and much vocabulary with Urdu, but it is NOT Urdu: it has its own spelling conventions, words, and pronunciation marks that sometimes differ from standard Urdu.\n\nYour only job is transcription. Not translation, not summarization, not correction.\n\nRules:\n\n1. Transcribe exactly what is printed on the page, in its original script, preserving line breaks as closely as you can to the visual layout.\n\n2. Do not translate anything into English or any other language. If a line is in English on the page, transcribe that line in English; if it is in Nastaliq, transcribe it in Nastaliq. Pages may mix the two (for example, dictionary entries with Nastaliq headwords and English definitions) \u2014 transcribe both exactly as they appear.\n\n3. Do not \"fix\" or normalize spelling toward standard Urdu. If a word looks unusual or different from how you would expect it in Urdu, transcribe it exactly as written. This is expected: Gojri regularly differs from Urdu even though it uses the same alphabet.\n\n4. Preserve diacritics and small marks exactly as shown (dots, hamzas, nasalization strokes, etc.) even if they look unusual. These marks carry real meaning in Gojri and are the single most error-prone part of this task \u2014 pay close attention to them.\n\n5. Include page numbers, running headers, footers, and captions exactly as they appear, in their normal reading position in your output.\n\n6. If part of the page is a photo, illustration, decorative border, or other non-text element, note it briefly instead of transcribing text that is not there. If you can clearly tell what it shows, say so in a short phrase, for example [illustration: a man handing cloth to a kneeling boy, others looking on] or [photo: a white rabbit]. If it is unclear, ambiguous, or you are not confident what it depicts, use a generic note instead, for example [illustration] or [photo], rather than guessing at details. Either way keep it to a short phrase, not a paragraph, and never invent text that is not actually printed on the page.\n\n7. If a word or short phrase is illegible, smudged, or genuinely ambiguous between more than one reading, make your single best attempt and wrap ONLY that word or phrase in double brackets, for example [[best guess]]. Do not silently guess without flagging it, and do not leave a blank gap instead of attempting a reading.\n\n8. If the page (or a large region of it) is blank, say so briefly, for example [blank] \u2014 do not invent content to fill empty space.\n\n9. This image is assumed to be a single logical page. If it clearly contains two full pages side by side (a spread that was not pre-split), transcribe the right-hand page in full first, then the left-hand page, since Gojri is read right-to-left.\n\n10. Output plain text only. No markdown formatting, no headings, no commentary before or after, no explanation of what you did or how confident you are as a separate summary. Just the transcription itself, using the bracket conventions above inline where needed.\n\nNow transcribe the page image that follows.\n"
print(PROMPT_V1)


In [ ]:
from transformers import AutoModel, AutoTokenizer
import torch

model_name = 'deepseek-ai/DeepSeek-OCR'
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)

try:
    model = AutoModel.from_pretrained(
        model_name, _attn_implementation='flash_attention_2',
        trust_remote_code=True, use_safetensors=True,
    )
    print("Loaded with flash_attention_2")
except Exception as e:
    print(f"flash_attention_2 failed ({e}), falling back to eager attention")
    model = AutoModel.from_pretrained(
        model_name, _attn_implementation='eager',
        trust_remote_code=True, use_safetensors=True,
    )

model = model.eval().cuda().to(torch.bfloat16)


In [ ]:
# The model's own prompt format is "<image>\n" followed by the instruction text -
# our full v1 prompt goes in as that instruction, unlike the Gradio demos' fixed
# task buttons, so this is a genuinely controlled, comparable run.
prompt = "<image>\n" + PROMPT_V1

for image_file in ["dict_alif.png", "gojri_adbiyaat.png"]:
    print(f"\n{'='*20} {image_file} {'='*20}\n")
    result = model.infer(
        tokenizer, prompt=prompt, image_file=image_file,
        output_path="./", base_size=1024, image_size=640,
        crop_mode=True, save_results=False,
    )
    print(result)
